# 02 · Tools and the MCP server

**One function, two transports.** `tool_function(source)` wraps `source.search` so it returns JSON text
while keeping its typed signature. That single function becomes:

```
                         ┌──► LangChain StructuredTool   (tools.mode: direct, in-process)
source.search ─► tool ───┤
                         └──► MCP tool on FastMCP        (tools.mode: mcp, any MCP client can use it)
```

**What you will learn**
1. How a typed Python signature becomes the JSON schema the LLM sees
2. The MCP server's tools, resource and prompt, in-process
3. MCP over HTTP, the way the Docker setup runs it

In [ ]:
# Setup: make the project importable and quiet the HTTP logs
import sys, logging, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
logging.getLogger("httpx").setLevel(logging.WARNING)

def show(obj, limit=1500):
    """Pretty-print a pydantic model, dict or list, cut to `limit` characters."""
    if hasattr(obj, "model_dump"):
        obj = obj.model_dump()
    if isinstance(obj, list):
        obj = [o.model_dump() if hasattr(o, "model_dump") else o for o in obj]
    text = json.dumps(obj, indent=2, ensure_ascii=False, default=str)
    print(text if len(text) <= limit else text[:limit] + "\n... (cut)")

In [ ]:
from pharma_agent.config import load_config
from pharma_agent.wiring import build_http, build_store
from pharma_agent.sources import build_sources
from pharma_agent.tools import tool_function, langchain_tools, parse_evidence
import inspect

cfg = load_config(overrides={"store.path": ":memory:"})
sources = build_sources(cfg["sources"], build_http(cfg, build_store(cfg)))

## 1. The wrapped function keeps the typed signature

In [ ]:
fn = tool_function(sources[0])
print(fn.__name__)
print(inspect.signature(fn))

## 2. As LangChain tools: the argument schema comes from that signature

In [ ]:
tools = langchain_tools(sources)
for t in tools:
    print(t.name)
    for arg, spec in t.args.items():
        print(f"    {arg}: {spec.get('type', spec.get('enum', ''))}  - {spec.get('description', '')}")

In [ ]:
text = tools[0].invoke({"drug": "warfarin", "section": "warnings"})
print(text[:400])
print("\nParsed back into Evidence:", [e.id for e in parse_evidence(text)])

## 3. The MCP server, in-process
`build_server()` registers the same functions on a FastMCP server, plus a **resource** (`pharma://sources`)
and a **prompt** (`drug_safety_review`). Here we call it directly, without a transport.

In [ ]:
from pharma_agent.mcp_server import build_server
server = build_server(cfg)

for t in await server.list_tools():
    print(f"{t.name:26s} args: {list(t.inputSchema['properties'])}")

In [ ]:
resource = await server.read_resource("pharma://sources")
print(list(resource)[0].content[:400])

In [ ]:
prompt = await server.get_prompt("drug_safety_review", {"drug": "adalimumab"})
print(prompt.messages[0].content.text)

## 4. MCP over HTTP (how Docker runs it)
Start the server as its own process, then connect with the same client library the agents use.

In [ ]:
import subprocess, socket, time

server_proc = subprocess.Popen(
    [sys.executable, "-m", "pharma_agent.mcp_server", "--transport", "streamable-http", "--port", "8765"],
    cwd=ROOT, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
)
for _ in range(60):  # wait until the port accepts connections
    try:
        socket.create_connection(("127.0.0.1", 8765), timeout=0.5).close(); break
    except OSError:
        time.sleep(0.5)
print("MCP server running, pid", server_proc.pid)

In [ ]:
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient({"pharma": {"transport": "streamable_http", "url": "http://127.0.0.1:8765/mcp"}})
mcp_tools = await client.get_tools()
print("Tools over MCP:", [t.name for t in mcp_tools])

result = await next(t for t in mcp_tools if t.name == "search_clinical_trials").ainvoke(
    {"intervention": "tirzepatide", "status": "RECRUITING", "limit": 2})
print([e.id for e in parse_evidence(result)])

In [ ]:
server_proc.terminate(); server_proc.wait()
print("MCP server stopped")

## Use these tools from Claude Desktop or Cursor
Add this to the client's MCP config (adjust the paths):
```json
{"mcpServers": {"pharma": {"command": "E:/pharma_agent/.venv/Scripts/python.exe",
                           "args": ["-m", "pharma_agent.mcp_server"], "cwd": "E:/pharma_agent"}}}
```
Next: **03_agents_and_graph.ipynb** gives these tools to LLM agents.